### Conversion from .nemo to .onnx models that can be used for on-device ASR inference

The script assumes that inside "{base_dir}/nemo_files/" folder there are .nemo ASR model files<br>
(normally these are CTC Conformer or FastConformer models as found e.g. on https://catalog.ngc.nvidia.com/search?orderBy=scoreDESC&query=ASR+Conformer or similar)

After having ran the script, the files that are of importance and used in final project are:

"{base_dir}/export/ASRMODEL_*.int8.onnx"

and

"{base_dir}/export/ASRVOCAB_*.txt"

In [ ]:
# POSSIBLE PREREQUISITES
# --------------------------
#!pip install causal-conv1d
#BRANCH = "r2.4.0"
#!pip install torch==2.4.0
#!pip install causal-conv1d>=1.4.0
#!pip install torch
#!pip install "nemo_toolkit[all] @ git+https://github.com/NVIDIA/NeMo.git@$BRANCH"

In [ ]:
import nemo
import nemo.collections.asr as nemo_asr
import torch
import io
import os
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

base_dir = "/mnt/e/uho_global_models_new/asr"

model_paths = [
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/teams/nemo/models/stt_fa_fastconformer_hybrid_large/files?version=1.21.0
    "stt_fa_fastconformer_hybrid_large.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/teams/nemo/models/stt_eo_conformer_ctc_large/files?version=1.14.0
    "stt_eo_conformer_ctc_large.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/teams/nemo/models/stt_ca_conformer_ctc_large/files?version=1.11.0
    "stt_ca_conformer_ctc_large.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/teams/nemo/models/stt_mr_conformer_ctc_medium/files?version=1.6.0
    "stt_mr_conformer_ctc_medium.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/teams/nemo/models/stt_rw_conformer_ctc_large?version=1.11.0
    "stt_rw_conformer_ctc_large.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/teams/riva/models/speechtotext_es_en_us_conformer_ml_cs/files?version=trainable_v1.1
    "Conformer-CTC-L_spe1024_ml_cs_es-en-US_1.1.nemo",
    # NOT https://catalog.ngc.nvidia.com/orgs/nvidia/teams/nemo/models/stt_ua_fastconformer_hybrid_large_pc/files?version=1.21.0
    # INSTEAD https://huggingface.co/nvidia/stt_ua_fastconformer_hybrid_large_pc/tree/main
    "stt_ua_fastconformer_hybrid_large_pc.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/teams/nemo/models/stt_pl_fastconformer_hybrid_large_pc/files?version=1.21.0
    "stt_pl_fastconformer_hybrid_large_pc.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/teams/riva/models/speechtotext_en_us_conformer/files?version=trainable_v6.0
    "Conformer-CTC-L_spe128_en-US_6.0.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/teams/nemo/models/stt_by_fastconformer_hybrid_large_pc/files?version=1.21.0
    "stt_by_fastconformer_hybrid_large_pc.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/teams/riva/models/speechtotext_ja_en_jp_conformer_unified_ml_cs/files?version=trainable_v7.0
    "Conformer-CTC-L-unified_char_ml_cs_ja-en-JP_7.0.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/teams/nemo/models/stt_fr_conformer_ctc_large/files?version=1.5.1
    "stt_fr_conformer_ctc_large.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/teams/riva/models/speechtotext_ar_ar_conformer/files?version=trainable_v3.0
    "Conformer-CTC-L_spe128_ar-AR_3.0.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/teams/riva/models/speechtotext_ko_kr_conformer/files?version=trainable_v1.0
    "Conformer-CTC-L_spe-2048_ko-KR_Riva-ASR-SET-1.0.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/teams/nemo/models/stt_ru_conformer_ctc_large?version=1.13.0
    "stt_ru_conformer_ctc_large.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/teams/riva/models/speechtotext_pt_br_conformer/files?version=trainable_v1.2
    "Conformer-CTC-L_bpe1024_pt-BR_1.2.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/teams/riva/models/speechtotext_zh_en_cn_conformer_ml_cs/files?version=trainable_v2.0
    "Conformer-CTC-L_spe7000_ml_cs_zh-en-CN_2.0.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/teams/nemo/models/stt_hr_conformer_ctc_large/files?version=1.11.0
    "stt_hr_conformer_ctc_large.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/teams/riva/models/speechtotext_hi_in_conformer/files?version=trainable_v3.0
    "Conformer-CTC-L_spe-128_hi-IN_Riva-ASR-SET-3.0.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/riva/models/speechtotext_nl_nl_conformer/-/file-browser
    "Conformer-CTC-L_spe256_nl-NL_1.0.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/riva/models/speechtotext_it_it_conformer/trainable_v2.1/file-browser
    "Conformer-CTC-L_spe1024_it-IT_2.1.nemo",
    # OK https://catalog.ngc.nvidia.com/orgs/nvidia/tao/models/speechtotext_de_de_conformer/trainable_v2.0/file-browser
    "Conformer-CTC-L_spe-1024_de-DE_Riva-ASR-SET-2.0.nemo",
    # OK https://www.clarin.si/repository/xmlui/handle/11356/1737
    "sl.nemo"
]

# NOTE: if you encounter an Out Of Memory error after several models having been processed,
# it is suggested to to change code in for loop below (... model_paths[:]) with index from which it should
# continue the next time you run it (e.g. ... model_paths[8:])

for input_path in model_paths[:]:
    full_input_path = os.path.join(base_dir, "nemo_files", input_path)
    onnx_path = os.path.join(base_dir, "export/ASRMODEL_" + input_path[0:-5] + ".onnx")
    tokens_path = os.path.join(base_dir, "export/ASRVOCAB_" + input_path[0:-5] + ".txt")
    #print(full_input_path, onnx_path, tokens_path)
    
    device = torch.device("cpu")
    model = nemo_asr.models.ASRModel.restore_from(full_input_path, map_location=device)
    #model = nemo_asr.models.EncDecCTCModelBPE.restore_from(input_path, map_location=device)
    try:
        model.change_decoding_strategy(decoder_type="ctc")
    except:
        pass

    # export onnx
    model.export(onnx_path)

    # quantize onnx
    input_model_path = onnx_path
    output_model_path = onnx_path.replace(".onnx", ".int8.onnx")

    quantize_dynamic(
            input_model_path,
            output_model_path,
            op_types_to_quantize=[
                'MatMul', 'Gemm',
                #'Conv',        # <-- doesn't work when quantizing Conv
                'Add', 'Mul'
            ],
            weight_type=QuantType.QInt8,    # signed 8-bit integer quantization
        )

    # create vocabulary txt files
    try:
        full_vocab = model.tokenizer.vocab
    except:
        print("(character based vocab)")
        full_vocab = model.decoder.vocabulary
    
    found_blank = False
    with io.open(tokens_path, mode="w") as f:
        for i in range(len(full_vocab)):
            word = full_vocab[i]
            if word == " ": # <-- edge case that may happen in character-based vocab (zh, jp, ...)
                word = "▁"
            if word == "":
                found_blank = True
                word = "<blk>"
            f.write("{} {}\n".format(word, i))
        
        if found_blank == False:
            f.write("<blk> {}".format(len(full_vocab)))
